In [1]:
# Cell 1: Setup and Path Routing
import sys
import os

# 1. Get the absolute path to the 'src' directory
# (Since the notebook is in /notebooks, we go up one level '..', then into 'src')
src_path = os.path.abspath(os.path.join('..', 'src'))

# 2. Add 'src' to the system path so Python knows where to look
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# 3. Enable autoreload so module changes reflect immediately
%load_ext autoreload
%autoreload 2

In [2]:
import os
from dotenv import load_dotenv

# Load API keys
load_dotenv()

# Import your newly modularized code
from schema import IntentSkeleton, ConnectionTarget
from prompts import get_intent_extraction_prompt
from graph import build_graph

In [3]:
# Initialize the graph
intent_graph = build_graph()


In [4]:
# Define 5 test queries for each Network Operating System
test_queries = {
    "Cisco IOS-XR": [
        # 1. Basic interface stats, seconds
        """Configure model-driven telemetry on my Cisco IOS-XR core router at 10.0.0.1:57400. Stream GigabitEthernet interface stats via dial-out gRPC to the collector at 172.16.1.5 port 50000 every 10 seconds.""",
        
        # 2. BGP state, hostname instead of IP, milliseconds
        """I need to set up streaming BGP neighbor telemetry for an XR device located at xr-spine-01:57400. Send it to a Splunk server (192.168.100.10:57000) using gRPC dial-out. The sample interval must be 5000ms.""",
        
        # 3. Conversational phrasing, Multiple Goals (CPU & Memory)
        """Hey, can you push CPU and memory metrics from my Cisco XR edge node (10.5.5.1 on port 57400) to our Telegraf instance at 10.10.10.100 port 57500? Use gRPC dial-out and a polling rate of 30 seconds.""",
        
        # 4. STRESS TEST: Explicit "null" strings for validator, Environmental goal
        """Set up MDT for power and temperature sensors on the XR router at 10.8.8.8. The collector is at 10.9.9.9 port 5432. The protocol is 'null' and encoding is 'null', let the system use the defaults. Poll every 60 seconds.""",
        
        # 5. STRESS TEST: Specific interfaces list, kv-gpb encoding, TCP protocol
        """On my IOS-XR device at 192.168.1.1:830, I want to monitor only HundredGigE0/0/0/0 and HundredGigE0/0/0/1. Stream this to 10.1.2.3:57000 using kv-gpb over TCP. Interval is 15000 milliseconds."""
    ],
    
    "Arista cEOS": [
        # 1. OpenConfig focus, milliseconds
        """Set up OpenConfig interface counters streaming on the Arista cEOS switch at clab-spine1:6030. Use gNMI dial-out to stream data to Prometheus at 192.168.50.20 port 9339. Sample every 10000 milliseconds.""",
        
        # 2. BGP routes, seconds
        """Configure the Arista cEOS device (10.1.1.2:6030) to stream BGP route updates to our collector at 10.99.0.5:50000. It needs to be a dial-out gRPC connection with a 2-second interval.""",
        
        # 3. Hardware environmentals, seconds
        """I want to monitor power and fan status on my cEOS leaf switch (leaf-01:6030). Push the telemetry via gRPC dial-out to 172.20.20.20 port 57000. The interval should be 15 seconds.""",
        
        # 4. STRESS TEST: Specific interfaces list, missing target port (should be None)
        """Can you configure my Arista switch at 10.50.50.50 to send telemetry for Ethernet1 and Ethernet2? I want to use dial-out gRPC to 10.100.100.100, but I don't know the collector port yet, so put null. Sample every 5 seconds.""",
        
        # 5. STRESS TEST: JSON Encoding, Multiple Goals, implicit device hints
        """Please set up the cEOS node arista-core-1:6030 to stream both CPU usage and Memory stats to 192.168.200.200:9999. Use JSON encoding over gRPC, dial-out mode, every 20 seconds."""
    ],
    
    "Nokia SR Linux": [
        # 1. Original variant, management interfaces
        """I need to monitor the management interface statistics on my Nokia SR Linux router at clab-quickstart-lab-r1:57400. Send the data via dial-out gRPC to my collector at 10.0.0.5 port 50000. Use a sample interval of 2000ms.""",
        
        # 2. System state, milliseconds
        """For the SR Linux node at 10.2.2.2:57400, configure a telemetry stream for system CPU and memory usage. Push it using gRPC to Telegraf at 10.0.5.50:57500 with a sample interval of 1000ms.""",
        
        # 3. Network Instances / BGP, seconds
        """Set up gRPC dial-out telemetry for BGP peer states on the Nokia SR Linux router at srl-edge-02:57400. The destination collector is 192.168.1.100 on port 57500, and the reporting interval should be 5 seconds.""",
        
        # 4. STRESS TEST: Missing target port, "other" telemetry goal (custom path)
        """Configure SR Linux router srl-leaf-01 to stream the custom path /network-instance/protocols/isis to our collector at 10.77.77.77:50000. Use a 10-second polling interval. The target router port is 'null', we'll use the default.""",
        
        # 5. STRESS TEST: Environmental, JSON Encoding, implicit hints
        """We need fan and temperature data from our SR Linux spine switch at 192.168.0.254:57400. Send it out to 10.254.254.254 port 60000 using JSON encoding. Make the sample interval 30 seconds."""
    ]
}


In [ ]:
# 1. Flatten the queries into a single list of inputs
flat_inputs = []
metadata = [] # Keep track of the OS and test number for printing later

for os_name, queries in test_queries.items():
    for i, query_text in enumerate(queries, 1):
        flat_inputs.append({"query": query_text.strip(), "skeleton": None})
        metadata.append({"os_name": os_name, "test_num": i, "query": query_text.strip()})

print(f"Starting parallel execution for {len(flat_inputs)} queries...")

# 2. Run everything in parallel using .batch()
# Note: You can optionally pass config={"max_concurrency": 5} to limit active threads
results = intent_graph.batch(flat_inputs)

# 3. Iterate through the results and print them beautifully
for meta, result in zip(metadata, results):
    print(f"\n{'#'*60}")
    print(f"### PLATFORM: {meta['os_name']} | Test {meta['test_num']} ###")
    print(f"{'#'*60}")
    print(f"Query: {meta['query']}\n")
    
    if result.get("skeleton"):
        print("Extracted JSON Skeleton:")
        # Depending on your object, you might need to call model_dump_json() or just print it
        print(result["skeleton"].model_dump_json(indent=2)) 
    else:
        print("❌ Graph failed to return a skeleton.")

Starting parallel execution for 15 queries...
